In [ ]:
import xarray as xr
import rioxarray as rxr
import matplotlib.pyplot as plt
from scipy.ndimage import label, generate_binary_structure
import numpy as np

In [ ]:
## Set base path

base = "/home/georg/data/LEON_P6_forest-agri-change/"

# Load small layer first (reference extent)
natural_forest = rxr.open_rasterio(base + "natural_forest_2020_bugoma.tif", masked=True)
forest_type_esri = rxr.open_rasterio(base + "LULC_ESRI_2019_Bugoma.tif", masked=True)

# Get bounds from reference
bounds = natural_forest.rio.bounds()

# Clip big layers to Bugoma extent (lazy loading)
forest_type_copernicus = rxr.open_rasterio(
    base + "PROBAV_LC100_global_v3.1.2_2019-nrt_Forest-Type-layer_EPSG-4326.tif",
    masked=True
).rio.clip_box(*bounds)

tree_density = rxr.open_rasterio(
    base + "Tree_cover_density_CLMS_2020/LCFM_TCD-10_V100_2020_N00E030_cog/LCFM_TCD-10_V100_2020_N00E030_MAP.tif",
    masked=True
).rio.clip_box(*bounds)

# Reproject all to EPSG:10793
forest_type_cop = forest_type_copernicus.rio.reproject("EPSG:10793")
forest_type_esri = forest_type_esri.rio.reproject("EPSG:10793")
natural_forest = natural_forest.rio.reproject("EPSG:10793")
tree_density = tree_density.rio.reproject("EPSG:10793")

print("Done")

In [ ]:
# Plot all used layers
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

forest_type_esri.plot(ax=axes[0], cmap='YlGn')
axes[0].set_title('LULC ESRI')

natural_forest.plot(ax=axes[1], cmap='Greens')
axes[1].set_title('Natural Forest')

tree_density.plot(ax=axes[2], cmap='RdYlGn')
axes[2].set_title('Tree Density')

plt.tight_layout()
plt.show()

In [ ]:
# Create forest mask

# Resample all to forest_type_esri grid (reference)
tree_density_matched = tree_density.rio.reproject_match(forest_type_esri)
natural_forest_matched = natural_forest.rio.reproject_match(forest_type_esri)

# Create forest mask
forest_mask = (
    (forest_type_esri == 2) & 
    (tree_density_matched > 30) & 
    (natural_forest_matched > 20)
).astype(int)


# Fill NaN with 0
forest_mask = forest_mask.fillna(0).astype('uint8')
forest_mask.rio.write_nodata(None, inplace=True)

In [ ]:
# Plot

fig, ax = plt.subplots(figsize=(10, 10))
forest_mask[0].plot(ax=ax, cmap='RdYlGn', cbar_kwargs={'label': 'Forest (1) / Non-Forest (0)'})
ax.set_title('Forest Extent')
plt.show()

In [ ]:
# Remove small patches (< 1 ha)

# Get pixel size in meters (for area calculation)
pixel_size_x, pixel_size_y = forest_mask.rio.resolution()
pixel_area_m2 = abs(pixel_size_x * pixel_size_y)
pixel_area_ha = pixel_area_m2 / 10000

# 8-connectivity structure (includes diagonals/corners)
structure = generate_binary_structure(2, 2)  # or np.ones((3,3))

# Label connected forest patches with 8-connectivity
labeled, num_patches = label(forest_mask.values[0] == 1, structure=structure)

# Calculate patch sizes in hectares
patch_sizes = np.bincount(labeled.ravel())
patch_sizes_ha = patch_sizes * pixel_area_ha

# Patches >= 1 ha (exclude label 0 = background)
valid_patches = np.where(patch_sizes_ha >= 1)[0]
valid_patches = valid_patches[valid_patches > 0]  # REMOVE BACKGROUND

# Create mask: keep only patches >= 1 ha
forest_filtered = np.isin(labeled, valid_patches).astype('uint8')

# Update forest_mask
forest_mask.values[0] = forest_filtered

print(f"Pixel area: {pixel_area_ha:.4f} ha")
print(f"Original patches: {num_patches}")
print(f"Patches >= 1 ha: {len(valid_patches)-1}")

# # Save
# forest_mask.rio.write_nodata(None, inplace=True)
# forest_mask.rio.to_raster(
#     "/home/georg/data/LEON_P6_forest-agri-change/forest_extent_bugoma_2019.tif",
#     compress='lzw'
# )

In [ ]:
# Plot

fig, ax = plt.subplots(figsize=(10, 10))
forest_mask[0].plot(ax=ax, cmap='RdYlGn', cbar_kwargs={'label': 'Forest (1) / Non-Forest (0)'})
ax.set_title('Forest Baseline 2020')
plt.show()

In [ ]:
forest_mask.rio.crs

In [ ]:
# Reproject and Save

fm = forest_mask.squeeze("band", drop=True)
fm_32636 = fm.rio.reproject("EPSG:32636")

fm_32636.rio.to_raster(
    "/home/georg/data/LEON_P6_forest-agri-change/forest_extent_bugoma_2020_32636.tif",
    compress="lzw"
)